In [1]:
import os
import pandas as pd

os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", None)
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

import tensorflow as tf

try:
    tf.random.set_seed(42)
except Exception:
    pass

output_dir = "./"
test_dir = "../input/plant-pathology-2021-fgvc8/test_images/"
model_dir = "../input/conve01/eff7-e17/epoch-17"
efficientB7 = "../input/efficientb7/effb7"

image_dims = (300, 300, 3)

data_set = pd.read_csv("../input/plant-pathology-2021-fgvc8/train.csv")
df_labels = data_set["labels"]

dataset_labels = [
    "scab",
    "frog_eye_leaf_spot",
    "rust",
    "complex",
    "powdery_mildew",
    "healthy",
]



2026-05-13 10:32:29.437559: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-13 10:32:29.454527: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778668349.477734      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778668349.484838      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-13 10:32:29.516470: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
from keras import Sequential, Model
from keras.layers import (
    Dense,
    BatchNormalization,
    Dropout,
    GlobalMaxPool2D,
    Conv2D,
    InputLayer,
)
from tensorflow import concat


def _find_savedmodel_dir_fast(path: str) -> str:
    if path is None:
        raise FileNotFoundError("Backbone path is None")
    path = os.path.expanduser(str(path))
    if os.path.isdir(path) and os.path.exists(os.path.join(path, "saved_model.pb")):
        return path
    raise FileNotFoundError(f"No saved_model.pb under: {path}")


def _resolve_backbone_dir_fast(preferred_path: str) -> str:
    try:
        return _find_savedmodel_dir_fast(preferred_path)
    except Exception:
        pass

    candidates = [
        preferred_path,
        os.path.join(preferred_path, "saved_model"),
        os.path.join(preferred_path, "model"),
        "../input/efficientb7/effb7",
        "../input/efficientb7",
        "../input/effb7",
    ]
    for c in candidates:
        try:
            return _find_savedmodel_dir_fast(c)
        except Exception:
            continue

    raise FileNotFoundError(
        f"Could not resolve EfficientNetB7 SavedModel from preferred={preferred_path} or common locations."
    )


class MultiLabel(Model):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        backbone_layer = None
        try:
            backbone_path = _resolve_backbone_dir_fast(efficientB7)
            if hasattr(tf.keras.layers, "TFSMLayer"):
                backbone_layer = tf.keras.layers.TFSMLayer(
                    backbone_path, call_endpoint="serving_default"
                )
            else:
                backbone_layer = None
        except Exception:
            backbone_layer = None

        self.model = Sequential()
        self.model.add(InputLayer(input_shape=image_dims))

        if backbone_layer is not None:

            class _SelectTensor(tf.keras.layers.Layer):
                def call(self, x):
                    y = backbone_layer(x)
                    if isinstance(y, dict):
                        y = y[sorted(y.keys())[0]]
                    return y

            self.model.add(_SelectTensor())
        else:
            backbone = tf.keras.applications.EfficientNetB7(
                include_top=False,
                weights="imagenet",
                input_shape=image_dims,
                pooling=None,
            )
            backbone.trainable = False
            self.model.add(backbone)

        self.model.add(Conv2D(filters=1024, kernel_size=(1, 1), padding="same"))
        self.model.add(BatchNormalization(momentum=0.7))
        self.model.add(Dropout(0.2))
        self.model.add(Conv2D(filters=1024, kernel_size=(1, 1), padding="same"))
        self.model.add(Conv2D(filters=2400, kernel_size=(1, 1), padding="same"))

        self.model_pred1_2 = Conv2D(filters=1024, kernel_size=(1, 1), padding="same")
        self.model_pred1_3 = BatchNormalization()
        self.model_pred1_4 = Conv2D(filters=512, kernel_size=(1, 1), padding="same")
        self.model_pred1_5 = GlobalMaxPool2D()
        self.model_pred1_6 = Dense(units=1, activation="sigmoid")

        self.model_pred2_2 = Conv2D(filters=1024, kernel_size=(1, 1), padding="same")
        self.model_pred2_3 = BatchNormalization()
        self.model_pred2_4 = Conv2D(filters=512, kernel_size=(1, 1), padding="same")
        self.model_pred2_5 = GlobalMaxPool2D()
        self.model_pred2_6 = Dense(units=1, activation="sigmoid")

        self.model_pred3_2 = Conv2D(filters=1024, kernel_size=(1, 1), padding="same")
        self.model_pred3_3 = BatchNormalization()
        self.model_pred3_4 = Conv2D(filters=512, kernel_size=(1, 1), padding="same")
        self.model_pred3_5 = GlobalMaxPool2D()
        self.model_pred3_6 = Dense(units=1, activation="sigmoid")

        self.model_pred4_2 = Conv2D(filters=1024, kernel_size=(1, 1), padding="same")
        self.model_pred4_3 = BatchNormalization()
        self.model_pred4_4 = Conv2D(filters=512, kernel_size=(1, 1), padding="same")
        self.model_pred4_5 = GlobalMaxPool2D()
        self.model_pred4_6 = Dense(units=1, activation="sigmoid")

        self.model_pred5_2 = Conv2D(filters=1024, kernel_size=(1, 1), padding="same")
        self.model_pred5_3 = BatchNormalization()
        self.model_pred5_4 = Conv2D(filters=512, kernel_size=(1, 1), padding="same")
        self.model_pred5_5 = GlobalMaxPool2D()
        self.model_pred5_6 = Dense(units=1, activation="sigmoid")

        self.model_pred6_2 = Conv2D(filters=1024, kernel_size=(1, 1), padding="same")
        self.model_pred6_3 = BatchNormalization()
        self.model_pred6_4 = Conv2D(filters=512, kernel_size=(1, 1), padding="same")
        self.model_pred6_5 = GlobalMaxPool2D()
        self.model_pred6_6 = Dense(units=1, activation="sigmoid")

    def call(self, x, **kwargs):
        y = self.model(x)

        pred1 = y[:, :, :, :400]
        pred2 = y[:, :, :, 400:800]
        pred3 = y[:, :, :, 800:1200]
        pred4 = y[:, :, :, 1200:1600]
        pred5 = y[:, :, :, 1600:2000]
        pred6 = y[:, :, :, 2000:2400]

        pred1 = self.model_pred1_2(pred1)
        pred1 = self.model_pred1_3(pred1)
        pred1 = self.model_pred1_4(pred1)
        pred1 = self.model_pred1_5(pred1)
        pred1 = self.model_pred1_6(pred1)

        pred2 = self.model_pred2_2(pred2)
        pred2 = self.model_pred2_3(pred2)
        pred2 = self.model_pred2_4(pred2)
        pred2 = self.model_pred2_5(pred2)
        pred2 = self.model_pred2_6(pred2)

        pred3 = self.model_pred3_2(pred3)
        pred3 = self.model_pred3_3(pred3)
        pred3 = self.model_pred3_4(pred3)
        pred3 = self.model_pred3_5(pred3)
        pred3 = self.model_pred3_6(pred3)

        pred4 = self.model_pred4_2(pred4)
        pred4 = self.model_pred4_3(pred4)
        pred4 = self.model_pred4_4(pred4)
        pred4 = self.model_pred4_5(pred4)
        pred4 = self.model_pred4_6(pred4)

        pred5 = self.model_pred5_2(pred5)
        pred5 = self.model_pred5_3(pred5)
        pred5 = self.model_pred5_4(pred5)
        pred5 = self.model_pred5_5(pred5)
        pred5 = self.model_pred5_6(pred5)

        pred6 = self.model_pred6_2(pred6)
        pred6 = self.model_pred6_3(pred6)
        pred6 = self.model_pred6_4(pred6)
        pred6 = self.model_pred6_5(pred6)
        pred6 = self.model_pred6_6(pred6)

        return concat([pred1, pred2, pred3, pred4, pred5, pred6], axis=1)

    def create_model(self):
        return self.model




In [3]:
if __name__ == "__main__":
    try:
        tf.config.optimizer.set_jit(False)
    except Exception:
        pass

    try:
        tf.config.optimizer.set_experimental_options(
            {
                "layout_optimizer": True,
                "constant_folding": True,
                "shape_optimization": True,
                "remapping": True,
                "arithmetic_optimization": True,
            }
        )
    except Exception:
        pass

    try:
        tf.config.threading.set_inter_op_parallelism_threads(0)
        tf.config.threading.set_intra_op_parallelism_threads(0)
    except Exception:
        pass

    model = MultiLabel()
    model.build(input_shape=[None, image_dims[0], image_dims[1], image_dims[2]])

    loaded = False
    weight_candidates = [
        model_dir,
        model_dir + ".h5",
        model_dir + ".weights.h5",
        os.path.join(model_dir, "weights.h5"),
        os.path.join(model_dir, "model.weights.h5"),
        os.path.join(model_dir, "checkpoint"),
    ]
    for cand in weight_candidates:
        try:
            if os.path.exists(cand):
                model.load_weights(cand)
                loaded = True
                break
        except Exception:
            pass

    sub_template = pd.read_csv(
        "../input/plant-pathology-2021-fgvc8/sample_submission.csv"
    )
    images_path_list = sub_template["image"].tolist()

    AUTOTUNE = tf.data.AUTOTUNE
    test_dir_tf = tf.constant(test_dir)

    # CHANGE (timeout): keep exact decode/resize/math, but ensure it is traced once and
    # runs fully in TF graph; combines I/O + preprocess in a single parallel map.
    @tf.function
    def _load_and_preprocess_one(name):
        path = tf.strings.join([test_dir_tf, name])
        img_bytes = tf.io.read_file(path)
        img = tf.io.decode_jpeg(img_bytes, channels=3, dct_method="INTEGER_FAST")
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(
            img,
            [image_dims[0], image_dims[1]],
            method=tf.image.ResizeMethod.BILINEAR,
        )
        img = img * 255.0
        return name, img

    @tf.function(
        input_signature=[
            tf.TensorSpec(
                shape=[None, image_dims[0], image_dims[1], image_dims[2]],
                dtype=tf.float32,
            )
        ],
    )
    def _infer_batch(x):
        return model(x, training=False)

    batch_size = 64
    threshold = 0.7

    # CHANGE (timeout): move thresholding + bitmask coding + label lookup into TF graph.
    # This preserves identical semantics to the previous numpy code but avoids large
    # per-batch device->host transfers of float predictions and Python-side processing.
    use_k = 6  # model output width is 6 and competition has 6 labels

    label_subset = dataset_labels[:use_k]
    table_py = [""] * (1 << use_k)
    for m in range(1 << use_k):
        if m == 0:
            table_py[m] = "healthy"
        else:
            parts = [label_subset[i] for i in range(use_k) if (m >> i) & 1]
            table_py[m] = " ".join(parts) if parts else "healthy"

    table_tf = tf.constant(table_py, dtype=tf.string)  # (64,)
    bit_weights_tf = tf.constant([1 << i for i in range(use_k)], dtype=tf.int32)  # (6,)

    @tf.function(
        input_signature=[
            tf.TensorSpec(shape=[None, use_k], dtype=tf.float32),
        ],
    )
    def _preds_to_labels_tf(preds):
        mask = preds > tf.cast(threshold, preds.dtype)  # bool (B,6)
        codes = tf.reduce_sum(tf.cast(mask, tf.int32) * bit_weights_tf[None, :], axis=1)
        return tf.gather(table_tf, codes)  # (B,)

    options = tf.data.Options()
    options.experimental_deterministic = True
    try:
        options.experimental_optimization.map_fusion = True
        options.experimental_optimization.map_parallelization = True
        options.experimental_optimization.parallel_batch = True
        options.experimental_optimization.autotune_buffers = True
    except Exception:
        pass
    try:
        options.threading.private_threadpool_size = 0
        options.threading.max_intra_op_parallelism = 0
    except Exception:
        pass

    ds = tf.data.Dataset.from_tensor_slices(tf.constant(images_path_list))
    ds = ds.with_options(options)
    ds = ds.map(
        _load_and_preprocess_one, num_parallel_calls=AUTOTUNE, deterministic=True
    )
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)

    n_total = len(images_path_list)
    out_images = [None] * n_total
    out_labels = [None] * n_total
    write_idx = 0

    # CHANGE (timeout): only transfer small string tensors per batch instead of full float predictions.
    for names_b, images_b in ds:
        preds_b = _infer_batch(images_b)  # TF tensor (B,6)
        labels_b = _preds_to_labels_tf(preds_b)  # TF string tensor (B,)

        names_np = names_b.numpy().astype("U")
        labels_np = labels_b.numpy().astype("U")
        bsz = names_np.shape[0]

        out_images[write_idx : write_idx + bsz] = names_np.tolist()
        out_labels[write_idx : write_idx + bsz] = labels_np.tolist()
        write_idx += bsz

    csv_pd = pd.DataFrame({"image": out_images, "labels": out_labels})
    csv_pd.to_csv(os.path.join(output_dir, "submission.csv"), index=False)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
2026-05-13 10:32:37.658081: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


258076736/258076736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'multi_label', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
2026-05-13 10:42:06.924017: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
